# Resolve AI — Stage 6: Return/refund risk model

Trains a model that estimates **how likely an order is to end in a refund/return**, using the
**Pakistan's Largest E-Commerce Dataset** (Kaggle, Z. Usmani) named in the project proposal.

Compares a **Logistic Regression baseline**, **XGBoost** and **LightGBM**, and reports precision, recall, F1,
ROC-AUC and PR-AUC on a **time-based** held-out test set (train on older orders, test on newer ones).

**Only features that Resolve also has for live Shopify orders are used** (order total, items, quantity,
unit price, discount, payment method, date, the customer's earlier orders), so the model can score real orders.

### How to run
- **Google Colab:** no GPU needed (CPU runtime is fine) → **Runtime → Run all**.
- **Your laptop:** `pip install pandas numpy scikit-learn xgboost lightgbm matplotlib kagglehub jupyter`, then open this notebook in VS Code/Jupyter and run all cells.

The dataset downloads automatically from Kaggle. If that fails, download the CSV from
https://www.kaggle.com/datasets/zusmani/pakistans-largest-ecommerce-dataset and upload it when asked.

At the end `resolve_risk_model.zip` (a few MB) is saved/downloaded for the Resolve backend.

In [ ]:
# 1. Setup
!pip -q install xgboost lightgbm kagglehub scikit-learn
import os, json, glob, shutil, warnings
from datetime import date
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

In [ ]:
# 2. Settings — which order statuses count as "returned/refunded" (1) and "kept" (0).
#    Other statuses (canceled, pending, processing, ...) are excluded: their outcome is not a return.
POSITIVE_STATUSES = {"order_refunded", "refund", "closed", "exchange"}
NEGATIVE_STATUSES = {"complete", "received"}
TEST_FRACTION = 0.2        # newest 20% of orders are the test set

In [ ]:
# 3. Get the dataset: Kaggle download, or manual upload as a fallback
csv_path = None
LOCAL_NAME = "Pakistan Largest Ecommerce Dataset.csv"   # running on a laptop with the CSV next to this notebook
if os.path.exists(LOCAL_NAME):
    csv_path = LOCAL_NAME
    print("Using local file:", csv_path)
try:
    if csv_path:
        raise StopIteration
    import kagglehub
    folder = kagglehub.dataset_download("zusmani/pakistans-largest-ecommerce-dataset")
    found = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
    csv_path = max(found, key=os.path.getsize) if found else None
    print("Downloaded from Kaggle:", csv_path)
except StopIteration:
    pass
except Exception as exc:
    print("Kaggle download did not work:", exc)

if csv_path is None:
    try:
        from google.colab import files
        print("Upload the dataset CSV downloaded from Kaggle:")
        csv_path = next(iter(files.upload()))
    except ImportError:
        csv_path = input("Path to the Kaggle CSV on this computer: ").strip().strip('"')
print("Using", csv_path)

In [ ]:
# 4. Load and inspect
try:
    raw = pd.read_csv(csv_path, low_memory=False)
except UnicodeDecodeError:
    raw = pd.read_csv(csv_path, low_memory=False, encoding="latin-1")
raw.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in raw.columns]
raw = raw.loc[:, ~raw.columns.str.startswith("unnamed")]
raw = raw.dropna(how="all")
print(raw.shape)
print(raw.columns.tolist())
display(raw.head(3))
print("\nOrder status counts:")
print(raw["status"].astype(str).str.strip().str.lower().value_counts().to_string())

In [ ]:
# 5. Clean item rows and build ORDER-level records (Resolve scores whole orders)
def pick(*names):
    for n in names:
        if n in raw.columns:
            return n
    return None

COL = {
    "order": pick("increment_id", "order_id"),
    "customer": pick("customer_id", "customer_since"),
    "created": pick("created_at", "working_date"),
    "status": pick("status"),
    "price": pick("price"),
    "qty": pick("qty_ordered", "quantity"),
    "total": pick("grand_total"),
    "discount": pick("discount_amount"),
    "payment": pick("payment_method"),
}
print(COL)
assert all(COL[k] for k in ["order", "created", "status", "price", "qty", "total", "payment"]), "Unexpected dataset columns — send this output to Claude"

items = pd.DataFrame({
    "order": raw[COL["order"]].astype(str).str.strip(),
    "customer": raw[COL["customer"]].astype(str).str.strip() if COL["customer"] else "unknown",
    # dates look like 7/1/2016 (month/day/year); fall back to automatic parsing for anything else
    "created": pd.to_datetime(raw[COL["created"]], format="%m/%d/%Y", errors="coerce")
               .fillna(pd.to_datetime(raw[COL["created"]], errors="coerce")),
    "status": raw[COL["status"]].astype(str).str.strip().str.lower(),
    "price": pd.to_numeric(raw[COL["price"]], errors="coerce"),
    "qty": pd.to_numeric(raw[COL["qty"]], errors="coerce"),
    "total": pd.to_numeric(raw[COL["total"]], errors="coerce"),
    "discount": pd.to_numeric(raw[COL["discount"]], errors="coerce") if COL["discount"] else 0.0,
    "payment": raw[COL["payment"]].astype(str).str.strip().str.lower(),
}).dropna(subset=["created", "price", "qty", "total"])
items = items[(items["price"] >= 0) & (items["qty"] > 0) & (items["total"] >= 0)]
items["discount"] = items["discount"].fillna(0).clip(lower=0)

items["is_pos"] = items["status"].isin(POSITIVE_STATUSES)
items["is_final"] = items["is_pos"] | items["status"].isin(NEGATIVE_STATUSES)

orders = items.groupby("order").agg(
    customer=("customer", "first"), created=("created", "min"), payment=("payment", "first"),
    order_total=("total", "sum"), discount_total=("discount", "sum"), item_count=("price", "size"),
    total_qty=("qty", "sum"), max_unit_price=("price", "max"),
    any_pos=("is_pos", "any"), all_final=("is_final", "all"),
).reset_index()
# An order is labelled only when every item has a final outcome; label 1 if any item was refunded/returned
orders = orders[orders["all_final"]].drop(columns=["all_final"])
orders["label"] = orders["any_pos"].astype(int)
orders = orders.drop(columns=["any_pos"]).sort_values("created").reset_index(drop=True)
print(f"{len(orders):,} labelled orders | refund/return rate: {orders['label'].mean():.2%}")

In [ ]:
# 6. Features available at scoring time (the same ones Resolve can compute for a Shopify order)
def payment_group(name: str) -> str:
    """Shared keyword rules — the Resolve backend uses the same mapping for Shopify gateway names."""
    n = str(name).lower()
    if "cod" in n or "cash" in n:
        return "cod"
    if any(k in n for k in ["easypay", "easypaisa", "jazz", "wallet", "mcblite", "sadapay", "nayapay"]):
        return "wallet"
    if any(k in n for k in ["card", "bank", "alfalah", "apg", "payaxis", "gateway", "stripe", "shopify_payments", "internet"]):
        return "card_bank"
    if any(k in n for k in ["credit", "voucher", "marketing", "finance", "gift"]):
        return "store_credit"
    return "other"

PAYMENT_GROUPS = ["cod", "wallet", "card_bank", "store_credit", "other"]
orders["payment_group"] = orders["payment"].map(payment_group)

# Customer history uses ONLY earlier orders (no peeking into the future)
orders["customer_prior_orders"] = orders.groupby("customer").cumcount()
prior_refunds = orders.groupby("customer")["label"].cumsum() - orders["label"]
orders["customer_prior_refund_rate"] = np.where(orders["customer_prior_orders"] > 0,
                                                prior_refunds / orders["customer_prior_orders"].replace(0, 1), 0.0)
orders.loc[orders["customer"].isin(["unknown", "nan", ""]), ["customer_prior_orders", "customer_prior_refund_rate"]] = 0

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=frame.index)
    X["log_order_total"] = np.log1p(frame["order_total"])
    X["log_max_unit_price"] = np.log1p(frame["max_unit_price"])
    X["item_count"] = frame["item_count"]
    X["total_qty"] = frame["total_qty"]
    X["discount_ratio"] = frame["discount_total"] / (frame["order_total"] + frame["discount_total"]).replace(0, np.nan)
    X["discount_ratio"] = X["discount_ratio"].fillna(0).clip(0, 1)
    X["day_of_week"] = frame["created"].dt.dayofweek
    X["month"] = frame["created"].dt.month
    X["customer_prior_orders"] = frame["customer_prior_orders"]
    X["customer_prior_refund_rate"] = frame["customer_prior_refund_rate"]
    for g in PAYMENT_GROUPS:
        X[f"pay_{g}"] = (frame["payment_group"] == g).astype(int)
    return X

X_all, y_all = build_features(orders), orders["label"]
FEATURES = X_all.columns.tolist()
print(FEATURES)
print(orders["payment_group"].value_counts().to_string())

In [ ]:
# 7. Time-based split: oldest 80% → train (last 15% of it = validation), newest 20% → test
n = len(orders)
cut_test = int(n * (1 - TEST_FRACTION))
cut_val = int(cut_test * 0.85)
X_train, y_train = X_all.iloc[:cut_val], y_all.iloc[:cut_val]
X_val, y_val = X_all.iloc[cut_val:cut_test], y_all.iloc[cut_val:cut_test]
X_test, y_test = X_all.iloc[cut_test:], y_all.iloc[cut_test:]
for name, y, part in [("train", y_train, orders.iloc[:cut_val]), ("val", y_val, orders.iloc[cut_val:cut_test]), ("test", y_test, orders.iloc[cut_test:])]:
    print(f"{name:5s} {len(y):7,d} orders | positive rate {y.mean():.2%} | {part['created'].min():%Y-%m-%d} → {part['created'].max():%Y-%m-%d}")
pos_weight = (y_train == 0).sum() / max(1, (y_train == 1).sum())

In [ ]:
# 8. Train: baseline, XGBoost, LightGBM
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_fscore_support,
                             precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay)
import xgboost as xgb, lightgbm as lgb

models = {
    "logistic_regression (baseline)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "xgboost": xgb.XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                                 scale_pos_weight=pos_weight, eval_metric="aucpr", early_stopping_rounds=50,
                                 random_state=SEED, n_jobs=-1),
    "lightgbm": lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                                   colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED, n_jobs=-1, verbose=-1),
}
models["xgboost"].fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
models["lightgbm"].fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
models["logistic_regression (baseline)"].fit(X_train, y_train)
print("trained:", list(models))

In [ ]:
# 9. Evaluate on the held-out (newest) test orders. Decision threshold = best F1 on VALIDATION (never tuned on test).
def best_threshold(y_true, scores):
    p, r, t = precision_recall_curve(y_true, scores)
    f1 = 2 * p * r / np.clip(p + r, 1e-9, None)
    return float(t[np.nanargmax(f1[:-1])]) if len(t) else 0.5

rows, scores_test = [], {}
for name, model in models.items():
    thr = best_threshold(y_val, model.predict_proba(X_val)[:, 1])
    s = model.predict_proba(X_test)[:, 1]
    scores_test[name] = s
    pred = (s >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average="binary", zero_division=0)
    rows.append({"model": name, "threshold": round(thr, 3), "precision": p, "recall": r, "f1": f1,
                 "roc_auc": roc_auc_score(y_test, s), "pr_auc": average_precision_score(y_test, s)})
results = pd.DataFrame(rows).set_index("model").round(3)
print(f"Test positive rate (a random guess would get PR-AUC ≈ this): {y_test.mean():.3f}")
display(results)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, s in scores_test.items():
    RocCurveDisplay.from_predictions(y_test, s, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, s, name=name, ax=axes[1])
axes[0].set_title("ROC curve (test)"); axes[1].set_title("Precision–recall curve (test)")
plt.tight_layout(); plt.show()

In [ ]:
# 10. Pick the best model (highest PR-AUC — best for imbalanced data), show what drives it, define risk tiers
best_name = results.loc[results.index != "logistic_regression (baseline)", "pr_auc"].idxmax()
best = models[best_name]
print("Best model:", best_name)

importances = pd.Series(best.feature_importances_, index=FEATURES).sort_values()
importances.plot.barh(figsize=(7, 5), title=f"Feature importance — {best_name}"); plt.tight_layout(); plt.show()

# Calibrate on VALIDATION: class weighting inflates raw scores, so map them to real refund probabilities
from sklearn.isotonic import IsotonicRegression
val_raw = best.predict_proba(X_val)[:, 1]
calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(val_raw, y_val)
val_cal = calibrator.predict(val_raw)
test_cal = calibrator.predict(scores_test[best_name])

# Tiers by share of orders (on validation): riskiest 15% = high, next 25% = medium, rest = low
HIGH_SHARE, MEDIUM_SHARE = 0.15, 0.25
high_cut = float(np.quantile(val_cal, 1 - HIGH_SHARE))
medium_cut = float(np.quantile(val_cal, 1 - HIGH_SHARE - MEDIUM_SHARE))
def tier(s): return np.where(s >= high_cut, "high", np.where(s >= medium_cut, "medium", "low"))

test_tiers = pd.DataFrame({"tier": tier(test_cal), "label": y_test.values})
print(f"Calibrated tier cut-offs: medium ≥ {medium_cut:.3f}, high ≥ {high_cut:.3f}  (test base rate {y_test.mean():.3f})")
tier_table = test_tiers.groupby("tier")["label"].agg(orders="size", actual_refund_rate="mean").reindex(["low", "medium", "high"])
tier_table["share_of_orders"] = tier_table["orders"] / tier_table["orders"].sum()
tier_table["share_of_all_refunds_caught"] = test_tiers.groupby("tier")["label"].sum().reindex(tier_table.index) / test_tiers["label"].sum()
display(tier_table.round(3))
ConfusionMatrixDisplay.from_predictions(y_test, (test_cal >= high_cut).astype(int),
                                        display_labels=["kept", "refunded"], cmap="Greens")
plt.title(f"{best_name} — test confusion matrix (high tier = flagged)"); plt.show()

In [ ]:
# 11. Save the model + everything the Resolve backend needs to score Shopify orders the same way
OUT = "resolve_risk_model"
os.makedirs(OUT, exist_ok=True)
tag = "xgb" if best_name == "xgboost" else "lgbm"
version = f"risk-{tag}-{date.today():%Y%m%d}"
if best_name == "xgboost":
    best.save_model(f"{OUT}/model.json")
else:
    best.booster_.save_model(f"{OUT}/model.txt")
meta = {
    "model_version": version, "algorithm": best_name, "features": FEATURES, "payment_groups": PAYMENT_GROUPS,
    # Backend: score = interp(raw_model_score, calibration.x, calibration.y), then compare with tiers
    "calibration": {"x": calibrator.X_thresholds_.tolist(), "y": calibrator.y_thresholds_.tolist()},
    "tiers": {"medium": medium_cut, "high": high_cut}, "tier_table_test": tier_table.round(4).reset_index().to_dict(orient="records"),
    "positive_statuses": sorted(POSITIVE_STATUSES), "negative_statuses": sorted(NEGATIVE_STATUSES),
    "test_metrics": results.loc[best_name].to_dict(), "all_results": results.reset_index().to_dict(orient="records"),
    "train_orders": int(len(y_train)), "dataset": "Pakistan's Largest E-Commerce Dataset (Kaggle, Z. Usmani)",
}
with open(f"{OUT}/resolve_risk_model.json", "w") as f:
    json.dump(meta, f, indent=2, default=str)
results.to_csv(f"{OUT}/results.csv")
shutil.make_archive("resolve_risk_model", "zip", OUT)
print("Saved", version, "→ resolve_risk_model.zip")
try:
    from google.colab import files
    files.download("resolve_risk_model.zip")
except ImportError:
    print("Zip is in:", os.path.abspath("resolve_risk_model.zip"))